In [ ]:

# ============================================================
# CROSS-DOMAIN IMAGE COLORIZATION USING USER INPUT + GUI
# ============================================================

import sys
import cv2
import torch
import numpy as np

from PIL import Image
from PyQt5.QtWidgets import QApplication

from PyQt5.QtWidgets import (
    QApplication,
    QWidget,
    QLabel,
    QPushButton,
    QFileDialog,
    QVBoxLayout,
    QHBoxLayout,
    QComboBox,
    QMessageBox
)

from PyQt5.QtGui import (
    QPixmap,
    QImage
)

from PyQt5.QtCore import Qt

from transformers import (
    SegformerImageProcessor,
    SegformerForSemanticSegmentation
)

print("Loading SegFormer...")

processor = SegformerImageProcessor.from_pretrained(
    "nvidia/segformer-b2-finetuned-ade-512-512"
)

model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b2-finetuned-ade-512-512"
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)
model.eval()

print("Model Loaded Successfully")

DOMAIN_COLORS = {

    "Satellite": {
        0: (0,0,255),
        1: (0,255,0),
        2: (128,128,128),
        3: (165,42,42)
    },

    "Night Vision": {
        0: (30,30,150),
        1: (0,180,0),
        2: (100,100,100),
        3: (180,100,50)
    }
}

def segment_image(image):

    image_rgb = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )

    pil_image = Image.fromarray(image_rgb)

    inputs = processor(
        images=pil_image,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        outputs = model(**inputs)

    logits = outputs.logits

    upsampled_logits = torch.nn.functional.interpolate(
        logits,
        size=image.shape[:2],
        mode="bilinear",
        align_corners=False
    )

    prediction = upsampled_logits.argmax(
        dim=1
    )[0]

    return prediction.cpu().numpy()

def thermal_colorization(image):
       
    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )

    colored = cv2.applyColorMap(
        gray,
        cv2.COLORMAP_INFERNO
    )

    return colored

def infrared_colorization(image):

    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )

    colored = cv2.applyColorMap(
        gray,
        cv2.COLORMAP_JET
    )

    return colored

def grayscale_colorization(image):

    hsv = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2HSV
    )

    hsv[:,:,1] = 120

    return cv2.cvtColor(
        hsv,
        cv2.COLOR_HSV2BGR
    )

def satellite_colorization(image):

    mask = segment_image(image)

    colored = image.copy()

    colored[mask == 2] = [255,0,0]
    colored[mask == 4] = [0,255,0]
    colored[mask == 6] = [128,128,128]
    colored[mask == 12] = [165,42,42]

    return colored

def nightvision_colorization(image):

    mask = segment_image(image)

    colored = image.copy()

    colored[mask == 2] = [100,100,255]
    colored[mask == 4] = [0,255,0]
    colored[mask == 6] = [120,120,120]

    return colored

def process_image(image, domain):

    if domain == "Thermal":
        return thermal_colorization(image)

    elif domain == "Infrared":
        return infrared_colorization(image)

    elif domain == "Satellite":
        return satellite_colorization(image)

    elif domain == "Night Vision":
        return nightvision_colorization(image)

    elif domain == "Grayscale":
        return grayscale_colorization(image)

    return image

class ColorizationGUI(QWidget):

    def __init__(self):
        super().__init__()

        self.setWindowTitle("Cross-Domain Image Colorization")
        self.setGeometry(150, 50, 1350, 800)

        # --------------------------------------------------
        # Main Window Style
        # --------------------------------------------------
        self.setStyleSheet("""
            QWidget{
                background-color:#1e1e2f;
                color:white;
                font-size:15px;
            }

            QLabel{
                color:white;
                font-size:15px;
            }

            QPushButton{
                background-color:#0078d7;
                color:white;
                border-radius:15px;
                padding:12px;
                font-size:15px;
                font-weight:bold;
            }

            QPushButton:hover{
                background-color:#00a2ff;
            }

            QComboBox{
                background:white;
                color:black;
                padding:8px;
                border-radius:10px;
                font-size:14px;
            }
        """)

        self.original_image = None

        # ==========================================
        # Title
        # ==========================================
        self.title = QLabel(
            "Cross-Domain Image Colorization using SegFormer"
        )

        self.title.setAlignment(Qt.AlignCenter)

        self.title.setStyleSheet("""
            font-size:28px;
            font-weight:bold;
            color:#00d9ff;
            padding:15px;
        """)

        # ==========================================
        # Image Labels
        # ==========================================
        self.original_label = QLabel()
        self.result_label = QLabel()

        self.original_label.setFixedSize(550,500)
        self.result_label.setFixedSize(550,500)

        self.original_label.setAlignment(Qt.AlignCenter)
        self.result_label.setAlignment(Qt.AlignCenter)

        self.original_label.setText("Original Image")
        self.result_label.setText("Colorized Output")

        self.original_label.setStyleSheet("""
            background-color:white;
            color:black;
            border:3px solid #00d9ff;
            border-radius:20px;
        """)

        self.result_label.setStyleSheet("""
            background-color:white;
            color:black;
            border:3px solid #00ff80;
            border-radius:20px;
        """)

        # ==========================================
        # Combo Box
        # ==========================================
        self.domain_combo = QComboBox()

        self.domain_combo.addItems([
            "Grayscale",
            "Thermal",
            "Infrared",
            "Satellite",
            "Night Vision"
        ])

        self.domain_combo.setFixedHeight(45)

        # ==========================================
        # Buttons
        # ==========================================
        self.upload_btn = QPushButton("Upload Image")
        self.colorize_btn = QPushButton("Colorize")
        self.save_btn = QPushButton("Save Image")

        self.upload_btn.setStyleSheet("""
            QPushButton{
                background:#3a86ff;
                color:white;
                border-radius:15px;
                padding:12px;
                font-size:15px;
                font-weight:bold;
            }

            QPushButton:hover{
                background:#5aa3ff;
            }
        """)

        self.colorize_btn.setStyleSheet("""
            QPushButton{
                background:#06d6a0;
                color:white;
                border-radius:15px;
                padding:12px;
                font-size:15px;
                font-weight:bold;
            }

            QPushButton:hover{
                background:#24f5bd;
            }
        """)

        self.save_btn.setStyleSheet("""
            QPushButton{
                background:#ff006e;
                color:white;
                border-radius:15px;
                padding:12px;
                font-size:15px;
                font-weight:bold;
            }

            QPushButton:hover{
                background:#ff3d93;
            }
        """)

        # ==========================================
        # Status Label
        # ==========================================
        self.status_label = QLabel(
            "Ready"
        )

        self.status_label.setAlignment(Qt.AlignCenter)

        self.status_label.setStyleSheet("""
            color:#ffd166;
            font-size:16px;
            padding:10px;
        """)

        # ==========================================
        # Connections
        # ==========================================
        self.upload_btn.clicked.connect(self.upload_image)
        self.colorize_btn.clicked.connect(self.colorize_image)
        self.save_btn.clicked.connect(self.save_image)

        # ==========================================
        # Layouts
        # ==========================================
        image_layout = QHBoxLayout()

        image_layout.addWidget(self.original_label)
        image_layout.addWidget(self.result_label)

        button_layout = QHBoxLayout()

        button_layout.addWidget(self.domain_combo)
        button_layout.addWidget(self.upload_btn)
        button_layout.addWidget(self.colorize_btn)
        button_layout.addWidget(self.save_btn)

        main_layout = QVBoxLayout()

        main_layout.addWidget(self.title)
        main_layout.addLayout(image_layout)
        main_layout.addSpacing(15)
        main_layout.addLayout(button_layout)
        main_layout.addWidget(self.status_label)

        self.setLayout(main_layout)
        
    def upload_image(self):

        file_path, _ = QFileDialog.getOpenFileName(
        self,
        "Open Image",
        "",
        "Images (*.png *.jpg *.jpeg *.bmp)"
        )

        if not file_path:
          return

        self.original_image = cv2.imread(file_path)

        self.display_image(
          self.original_image,
          self.original_label
        )

        self.status_label.setText(
          "Image loaded successfully"
        )

    def colorize_image(self):

        if self.original_image is None:

          QMessageBox.warning(
            self,
            "Warning",
            "Please upload an image first."
          )
          return

        domain = self.domain_combo.currentText()

        result = process_image(
         self.original_image,
         domain
        )

        self.display_image(
         result,
         self.result_label
        )

        self.status_label.setText(
         "Colorization completed"
        )

    def display_image(self, image, label):

       rgb = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
       )

       h, w, ch = rgb.shape

       bytes_per_line = ch * w

       qt_img = QImage(
        rgb.data,
        w,
        h,
        bytes_per_line,
        QImage.Format_RGB888
       )

       pixmap = QPixmap.fromImage(qt_img)

       pixmap = pixmap.scaled(
        label.width(),
        label.height(),
        Qt.KeepAspectRatio
       )

       label.setPixmap(pixmap)

    def save_image(self):

        if self.result_label.pixmap() is None:
         QMessageBox.warning(
            self,
            "Warning",
            "No colorized image available."
         )
         return

        save_path, _ = QFileDialog.getSaveFileName(
         self,
         "Save Image",
         "",
         "PNG Files (*.png);;JPEG Files (*.jpg)"
        )

        if save_path:
         result = process_image(
            self.original_image,
            self.domain_combo.currentText()
         )

         cv2.imwrite(save_path, result)

         self.status_label.setText(
            "Image saved successfully"
         )

if __name__ == "__main__":

    app = QApplication(sys.argv)

    window = ColorizationGUI()

    window.show()

    sys.exit(app.exec_())



Loading SegFormer...


In [6]:
pip install transformers

   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.0 MB 2.9 MB/s eta 0:00:04
   --- ------------------------------------ 1.0/11.0 MB 2.3 MB/s eta 0:00:05
   ----- ---------------------------------- 1.6/11.0 MB 2.2 MB/s eta 0:00:05
   ------- -------------------------------- 2.1/11.0 MB 2.5 MB/s eta 0:00:04
   ----------- ---------------------------- 3.1/11.0 MB 2.9 MB/s eta 0:00:03
   -------------- ------------------------- 3.9/11.0 MB 3.0 MB/s eta 0:00:03
   ---------------- ----------------------- 4.5/11.0 MB 2.9 MB/s eta 0:00:03
   ------------------ --------------------- 5.0/11.0 MB 2.9 MB/s eta 0:00:03
   -------------------- ------------------- 5.5/11.0 MB 2.8 MB/s eta 0:00:02
   -------------------- ------------------- 5.8/11.0 MB 2.8 MB/s eta 0:00:02
   ----------------------- ---------------- 6.6/11.0 MB 2.8 MB/s eta 0:00:02
   ----------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


NameError: name 'outputs' is not defined